# Flatten instruments
Project normalized instrument lifecycle rows from the sorted market-log stream.

In [ ]:
source = "fixmessage.market"
target = "market.instruments"
start = None
end = None
catalog = "rekep"
catalog_properties = {}
branch = "root"
merge_by = True
commit_row_size = 250_000

In [ ]:
from pyiceberg.expressions import And, EqualTo, GreaterThanOrEqual, LessThan
from rekep.iceberg import IcebergDataset
from rekep.market import EventType, Instrument
from rekep.text import FixMessage
from rekep.times import unix_of

lower, upper = unix_of(start), unix_of(end, upper=True)


def _window(column="unix"):
    predicates = [
        EqualTo("etype", int(EventType.INSTRUMENT)),
        EqualTo("plugin_code", FixMessage.into_instrument_plugin()),
    ]
    if lower is not None:
        predicates.append(GreaterThanOrEqual(column, lower))
    if upper is not None:
        predicates.append(LessThan(column, upper))
    return predicates[0] if len(predicates) == 1 else And(*predicates)


logs_table = IcebergDataset(
    name=source, catalog=catalog, properties=dict(catalog_properties), branch=branch
)
instrument_table = IcebergDataset(
    name=target,
    catalog=catalog,
    properties=dict(catalog_properties),
    branch=branch,
    field=Instrument.into_field(),
    commit_row_size=commit_row_size,
    sort_by=("unix", "version", "hash"),
)


counts = {"versions": 0}


def _batches():
    reader = logs_table.read_arrow_reader(
        FixMessage.into_field(), row_filter=_window(), order_by=("unix", "msg_seq_num", "hash")
    )
    for batch in reader:
        instruments = FixMessage.into_instrument_arrow_batch(batch)
        counts["versions"] += instruments.num_rows
        if instruments.num_rows:
            yield instruments


written = instrument_table.append_arrow_reader(
    _batches(), Instrument.into_field(), merge_by=merge_by, commit_row_size=commit_row_size
)
result = {"versions": counts["versions"], "written": written, "target": target}
try:
    import scrapbook as sb
except ImportError:
    pass
else:
    sb.glue("result", result, encoder="json")
result